In [ ]:
# UNIVERSAL GPU CONFIGURATION FOR TENSORFLOW
# ================================================================
!pip install tensorflow[and-cuda]



import tensorflow as tf
import os

# Configure TensorFlow to use GPU automatically everywhere
def configure_gpu():
    # Get available GPUs
    gpus = tf.config.list_physical_devices('GPU')
    
    if gpus:
        try:
            # Enable memory growth to avoid allocating all memory at once
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            
            # Set logical GPU device configuration
            logical_gpus = tf.config.list_logical_devices('GPU')
            
            print(f"✅ GPU Configuration Successful!")
            print(f"📊 Found {len(gpus)} physical GPU(s), {len(logical_gpus)} logical GPU(s)")
            
            # Set GPU as default device for all operations
            tf.config.set_soft_device_placement(True)
            
            # Verify GPU is available and will be used
            print("🎯 TensorFlow will automatically use GPU for all operations")
            
        except RuntimeError as e:
            print(f"⚠️ GPU configuration error: {e}")
            print("🔧 Falling back to CPU")
    else:
        print("❌ No GPU found - Using CPU")
    
    return len(gpus) > 0

# Set environment variable to force GPU usage
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# Call the configuration function
gpu_available = configure_gpu()

# Optional: Set GPU memory limit (adjust as needed)
if gpu_available:
    try:
        # Limit to 8GB if you have memory issues
        tf.config.set_logical_device_configuration(
            gpus[0],
            [tf.config.LogicalDeviceConfiguration(memory_limit=8192)]
        )
        print("🔧 GPU memory limit set to 8GB")
    except:
        print("⚠️ Could not set memory limit - using default")

print("🚀 TensorFlow is ready! All operations will use GPU automatically.")

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
import pandas as pd
import numpy as np

folder = "dataunsup"

# Load dataset sizes (RAM tiny)
df_wel  = pd.read_csv(f"{folder}/welfake_clean.csv")
df_net  = pd.read_csv(f"{folder}/fakenewsnet_clean.csv")
df_pred = pd.read_csv(f"{folder}/news_clean.csv")

n_wel  = len(df_wel)
n_net  = len(df_net)
n_pred = len(df_pred)

MAX_LEN = 300
EMB_DIM = 300

# ✅ Load memmap safely (zero RAM usage)
X_wel_unsup = np.memmap(f"{folder}/welfake_unsup_seq.dat",
                        dtype="float32", mode="r",
                        shape=(n_wel, MAX_LEN, EMB_DIM))

X_net_unsup = np.memmap(f"{folder}/fakenewsnet_unsup_seq.dat",
                        dtype="float32", mode="r",
                        shape=(n_net, MAX_LEN, EMB_DIM))

X_pred_unsup = np.memmap(f"{folder}/fakepred_unsup_seq.dat",
                         dtype="float32", mode="r",
                         shape=(n_pred, MAX_LEN, EMB_DIM))

# ✅ y labels are small → load normally
y_wel  = np.load(f"{folder}/welfake_labels.npy")
y_net  = np.load(f"{folder}/fakenewsnet_labels.npy")
y_pred = np.load(f"{folder}/fakepred_labels.npy")

print("✅ All datasets loaded via memmap")


In [ ]:
def make_dataset(X, y, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    ds = ds.shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


In [ ]:
from tensorflow.keras import layers, models, regularizers

MAX_LEN = 300         # shape is (N, 300, 300)
EMB_DIM = 300
L2_LAMBDA = 0.01
NUM_CLASSES = 2


def build_cnn_lstm():
    model = models.Sequential([
        layers.Input(shape=(MAX_LEN, EMB_DIM)),
        
        layers.Conv1D(64, 4, activation='relu',
                      kernel_regularizer=regularizers.l2(L2_LAMBDA)),
        layers.Conv1D(64, 3, activation='relu',
                      kernel_regularizer=regularizers.l2(L2_LAMBDA)),
        
        layers.MaxPooling1D(pool_size=2),
        
        layers.LSTM(50, return_sequences=True,
                    kernel_regularizer=regularizers.l2(L2_LAMBDA)),
        layers.LSTM(30, kernel_regularizer=regularizers.l2(L2_LAMBDA)),
        
        # Keep 2 outputs but use softmax
        layers.Dense(2, activation='softmax')  # Use softmax for multi-class
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',  # Use this for 1D integer labels
        metrics=['accuracy']
    )
    return model


In [ ]:
def train_model(X, y, name, batch_size=32, epochs=10, save_dir="dataunsup"):
    import tensorflow as tf
    import numpy as np
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support
    import os

    print(f"\n🚀 Training {name} using GPU streaming...")

    # Ensure lengths match
    min_len = min(len(X), len(y))
    X, y = X[:min_len], y[:min_len]

    # Split only indices (not full arrays)
    idx_train, idx_test = train_test_split(
        np.arange(len(y)), test_size=0.2, random_state=42, stratify=y
    )

    y_train, y_test = y[idx_train], y[idx_test]
    print(f"📊 Train size: {len(idx_train)} | Test size: {len(idx_test)}")

    # Generator to stream batches directly from memmap
    def data_gen(idxs, batch_size):
        n = len(idxs)
        while True:
            np.random.shuffle(idxs)
            for i in range(0, n, batch_size):
                batch_idx = idxs[i:i+batch_size]
                yield X[batch_idx], y[batch_idx]

    # TensorFlow Datasets (GPU-optimized streaming)
    ds_train = tf.data.Dataset.from_generator(
        lambda: data_gen(idx_train, batch_size),
        output_signature=(
            tf.TensorSpec(shape=(None, 300, 300), dtype=tf.float32),
            tf.TensorSpec(shape=(None,), dtype=tf.int64)
        )
    ).prefetch(tf.data.AUTOTUNE)

    ds_test = tf.data.Dataset.from_generator(
        lambda: data_gen(idx_test, batch_size),
        output_signature=(
            tf.TensorSpec(shape=(None, 300, 300), dtype=tf.float32),
            tf.TensorSpec(shape=(None,), dtype=tf.int64)
        )
    ).prefetch(tf.data.AUTOTUNE)

    # Build CNN-LSTM model
    model = build_cnn_lstm()

    # Train safely on GPU
    history = model.fit(
        ds_train,
        validation_data=ds_test,
        steps_per_epoch=len(idx_train)//batch_size,
        validation_steps=len(idx_test)//batch_size,
        epochs=epochs,
        verbose=1
    )

    # Evaluate model
    preds = model.predict(ds_test, steps=len(idx_test)//batch_size)
    y_pred = np.argmax(preds, axis=1)

    acc = accuracy_score(y_test[:len(y_pred)], y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_test[:len(y_pred)], y_pred, average='binary'
    )

    print(f"\n📊 Results for {name}:")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 Score : {f1:.4f}")

    # Save model
    os.makedirs(save_dir, exist_ok=True)
    model_path = os.path.join(save_dir, f"cnn_lstm_{name}.h5")
    model.save(model_path)

    print(f"✅ Model saved at: {model_path}")

    # Save metrics for later comparison
    metrics_path = os.path.join(save_dir, f"{name}_metrics.txt")
    with open(metrics_path, "w") as f:
        f.write(f"Accuracy : {acc:.4f}\n")
        f.write(f"Precision: {prec:.4f}\n")
        f.write(f"Recall   : {rec:.4f}\n")
        f.write(f"F1 Score : {f1:.4f}\n")

    print(f"📁 Metrics saved at: {metrics_path}")

    return history, model_path


In [7]:
train_model(X_wel_unsup, y_wel, "welfake_unsup")


🚀 Training welfake_unsup using GPU streaming...
📊 Train size: 57707 | Test size: 14427
Epoch 1/10


2025-11-12 21:03:31.053629: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91500


1803/1803 ━━━━━━━━━━━━━━━━━━━━ 70s 37ms/step - accuracy: 0.5321 - loss: 0.7754 - val_accuracy: 0.5145 - val_loss: 0.6931
Epoch 2/10
1803/1803 ━━━━━━━━━━━━━━━━━━━━ 56s 31ms/step - accuracy: 0.5121 - loss: 0.6930 - val_accuracy: 0.5144 - val_loss: 0.6929
Epoch 3/10
1803/1803 ━━━━━━━━━━━━━━━━━━━━ 55s 31ms/step - accuracy: 0.5133 - loss: 0.6929 - val_accuracy: 0.5144 - val_loss: 0.6927
Epoch 4/10
1803/1803 ━━━━━━━━━━━━━━━━━━━━ 84s 47ms/step - accuracy: 0.5146 - loss: 0.6928 - val_accuracy: 0.5144 - val_loss: 0.6934
Epoch 5/10
1803/1803 ━━━━━━━━━━━━━━━━━━━━ 59s 33ms/step - accuracy: 0.5142 - loss: 0.6928 - val_accuracy: 0.5147 - val_loss: 0.6929
Epoch 6/10
1803/1803 ━━━━━━━━━━━━━━━━━━━━ 59s 33ms/step - accuracy: 0.5145 - loss: 0.6928 - val_accuracy: 0.5142 - val_loss: 0.6928
Epoch 7/10
1803/1803 ━━━━━━━━━━━━━━━━━━━━ 60s 33ms/step - accuracy: 0.5143 - loss: 0.6928 - val_accuracy: 0.5145 - val_loss: 0.6927
Epoch 8/10
1803/1803 ━━━━━━━━━━━━━━━━━━━━ 61s 34ms/step - accuracy: 0.5140 - loss: 0.69


📊 Results for welfake_unsup:
Accuracy : 0.5144
Precision: 0.5144
Recall   : 1.0000
F1 Score : 0.6793
✅ Model saved at: dataunsup/cnn_lstm_welfake_unsup.h5
📁 Metrics saved at: dataunsup/welfake_unsup_metrics.txt


(<keras.src.callbacks.history.History at 0x7f22b6188dc0>,
 'dataunsup/cnn_lstm_welfake_unsup.h5')

In [8]:
train_model(X_net_unsup, y_net, "fakenewsnet_unsup")


🚀 Training fakenewsnet_unsup using GPU streaming...
📊 Train size: 18556 | Test size: 4640
Epoch 1/10
579/579 ━━━━━━━━━━━━━━━━━━━━ 19s 30ms/step - accuracy: 0.7505 - loss: 0.7870 - val_accuracy: 0.7519 - val_loss: 0.5606
Epoch 2/10
579/579 ━━━━━━━━━━━━━━━━━━━━ 17s 28ms/step - accuracy: 0.7519 - loss: 0.5616 - val_accuracy: 0.7519 - val_loss: 0.5622
Epoch 3/10
579/579 ━━━━━━━━━━━━━━━━━━━━ 16s 28ms/step - accuracy: 0.7519 - loss: 0.5606 - val_accuracy: 0.7519 - val_loss: 0.5606
Epoch 4/10
579/579 ━━━━━━━━━━━━━━━━━━━━ 16s 28ms/step - accuracy: 0.7521 - loss: 0.5606 - val_accuracy: 0.7519 - val_loss: 0.5612
Epoch 5/10
579/579 ━━━━━━━━━━━━━━━━━━━━ 16s 28ms/step - accuracy: 0.7519 - loss: 0.5608 - val_accuracy: 0.7519 - val_loss: 0.5621
Epoch 6/10
579/579 ━━━━━━━━━━━━━━━━━━━━ 16s 28ms/step - accuracy: 0.7516 - loss: 0.5611 - val_accuracy: 0.7519 - val_loss: 0.5650
Epoch 7/10
579/579 ━━━━━━━━━━━━━━━━━━━━ 16s 28ms/step - accuracy: 0.7513 - loss: 0.5617 - val_accuracy: 0.7519 - val_loss: 0.5604


📊 Results for fakenewsnet_unsup:
Accuracy : 0.7519
Precision: 0.7519
Recall   : 1.0000
F1 Score : 0.8584
✅ Model saved at: dataunsup/cnn_lstm_fakenewsnet_unsup.h5
📁 Metrics saved at: dataunsup/fakenewsnet_unsup_metrics.txt


(<keras.src.callbacks.history.History at 0x7f225825bbe0>,
 'dataunsup/cnn_lstm_fakenewsnet_unsup.h5')

In [9]:
train_model(X_pred_unsup, y_pred, "newspred_unsup")


🚀 Training newspred_unsup using GPU streaming...
📊 Train size: 5068 | Test size: 1267
Epoch 1/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - accuracy: 0.8975 - loss: 1.1768 - val_accuracy: 0.9022 - val_loss: 0.4044
Epoch 2/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.9015 - loss: 0.3479 - val_accuracy: 0.9022 - val_loss: 0.3255
Epoch 3/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.9011 - loss: 0.3267 - val_accuracy: 0.9006 - val_loss: 0.3246
Epoch 4/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.9015 - loss: 0.3230 - val_accuracy: 0.9022 - val_loss: 0.3203
Epoch 5/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.9031 - loss: 0.3190 - val_accuracy: 0.8998 - val_loss: 0.3255
Epoch 6/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.8997 - loss: 0.3268 - val_accuracy: 0.9014 - val_loss: 0.3231
Epoch 7/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - accuracy: 0.9009 - loss: 0.3237 - val_accuracy: 0.9030 - val_loss: 0.3213
Epoch 8/10

/home/nasc/ak/env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



📊 Results for newspred_unsup:
Accuracy : 0.9006
Precision: 0.0000
Recall   : 0.0000
F1 Score : 0.0000
✅ Model saved at: dataunsup/cnn_lstm_newspred_unsup.h5
📁 Metrics saved at: dataunsup/newspred_unsup_metrics.txt


(<keras.src.callbacks.history.History at 0x7f2248370be0>,
 'dataunsup/cnn_lstm_newspred_unsup.h5')